# Benchmark evidence

InferDoc measures a bounded workload. Non-streaming is the default because it can retain complete provider token usage. The fallback below loads a sanitized measured run when the live gate is off.

In [ ]:
import asyncio
import os
from pathlib import Path
import inferdoc
from inferdoc.config import InferDocSettings
from inferdoc.evidence.models import EvidenceBundle

settings = InferDocSettings.from_env()
live_enabled = os.getenv("INFERDOC_RUN_LIVE_TESTS") == "1" and bool(settings.api_key)
prompts = ["Define TTFT in one sentence.", "Define throughput in one sentence."]
print({"model": settings.default_model, "live_enabled": live_enabled})

## Measure or load a captured run

This calls the real benchmark SDK only with explicit live opt-in. `enable_thinking=False` applies to workload requests.

In [ ]:
if live_enabled:
    run = await asyncio.to_thread(
        inferdoc.benchmark, model=settings.default_model, prompts=prompts,
        concurrency=1, max_tokens=32, stream=False, enable_thinking=False,
    )
    source = "new live run"
else:
    run = EvidenceBundle.model_validate_json(
        Path("examples/demo_artifacts/baseline.json").read_text(encoding="utf-8")
    )
    source = "sanitized captured run"
print({"source": source, "run_id": run.run_id, "request_count": len(run.requests)})

## Deterministic aggregates and provenance

Unavailable values remain `None`. TTFT is unavailable for non-streaming evidence. GPU and KV-cache telemetry are not exposed by this serverless run.

In [ ]:
aggregate = run.aggregate
print({"success_rate": aggregate["success_rate"],
       "input_tokens": aggregate["total_input_tokens"],
       "output_tokens": aggregate["total_output_tokens"],
       "total_tokens": aggregate["total_tokens"],
       "latency_s": aggregate["latency_s"],
       "request_throughput_rps": aggregate["request_throughput_rps"],
       "output_token_throughput_tps": aggregate["output_token_throughput_tps"],
       "ttft_s": aggregate["ttft_s"]})
print("run-level capabilities:", run.capabilities.model_dump())
print("provenance hashes:", {key: value for key, value in run.provenance.items() if key.endswith("sha256")})

## Optional streaming comparison

Set `RUN_STREAMING_DEMO=True` and the live gate to spend credits on a second run. Streaming may observe TTFT when a first chunk arrives. Current streaming code does not retain provider token usage, so token-throughput metrics may be unavailable.

In [ ]:
RUN_STREAMING_DEMO = False
streamed = None
if RUN_STREAMING_DEMO and live_enabled:
    streamed = await asyncio.to_thread(
        inferdoc.benchmark, model=settings.default_model, prompts=prompts,
        concurrency=1, max_tokens=32, stream=True, enable_thinking=False,
    )
    print({"ttft_s": streamed.aggregate["ttft_s"],
           "token_usage_available": streamed.capabilities.token_usage_available,
           "output_token_throughput_tps": streamed.aggregate["output_token_throughput_tps"]})
else:
    print("Optional streaming comparison skipped.")